# LongMemEval n=100 — merken on paraphrase-multilingual

Built 2026-04-14. Lightweight companion run to the 2026-04-13 bge-small-en full-split baseline (R@5=0.964 on n=500).

**Question this answers:** does swapping the embedder from `BAAI/bge-small-en-v1.5` to `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` regress on English-only LongMemEval?

**Why n=100 and not n=500:** the original CI at n=500 is ±0.016. At n=100 it widens to ~±0.04 — still tight enough to detect a real regression, and ~5x cheaper to run.

**Runtime (CPU, A100 node):** ~45 min. No CUDA wiring needed — for a 384-dim MiniLM model the CPU throughput on Colab is adequate and removes a whole class of Colab/CUDA version mismatches.

**Instructions:**
1. Runtime → any runtime works; A100 is fine if you have it, but not required.
2. Run all cells top to bottom. Cell 1 clones and installs; cell 2 pins the embedder; cell 3 runs a ~2 min sanity; cell 4 runs the full n=100.
3. Copy the final `baseline=merken-heuristic` line into `RESULTS.md`.

In [1]:
# 1. Clone repos and install merken + vstash (editable).
#
# We do NOT touch onnxruntime here — fastembed ships its own and the
# bilingual probe does not need GPU. Fewer moving parts = fewer failures.

!git clone https://github.com/stffns/vstash.git /content/vstash 2>/dev/null || (cd /content/vstash && git pull)
!git clone https://github.com/stffns/merken.git /content/merken 2>/dev/null || (cd /content/merken && git pull)

%pip install -e /content/vstash -q
%pip install -e /content/merken -q

# Fallback: add source paths directly so Python finds the packages even
# if the pip editable metadata is not picked up mid-session.
import sys
for p in ('/content/vstash', '/content/merken'):
    if p not in sys.path:
        sys.path.insert(0, p)

import vstash, merken
print(f'vstash: {vstash.__file__}')
print(f'merken: {merken.__file__}')

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 104.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 36.4 MB/s eta 0:00:00
  Building editable for vstash (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for merken

In [2]:
# 2. Pin the multilingual embedder via VSTASH_CONFIG.
import os, tempfile
from pathlib import Path

CFG_DIR = Path(tempfile.mkdtemp(prefix='merken_colab_'))
CFG_PATH = CFG_DIR / 'vstash.toml'
CFG_PATH.write_text(
    '[embeddings]\n'
    'model = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"\n'
    'backend = "auto"\n'
)
os.environ['VSTASH_CONFIG'] = str(CFG_PATH)
print(f'VSTASH_CONFIG = {os.environ["VSTASH_CONFIG"]}')

from vstash.config import load_config
cfg = load_config()
assert cfg.embeddings.model.endswith('paraphrase-multilingual-MiniLM-L12-v2'), \
    f'unexpected model: {cfg.embeddings.model}'
print(f'vstash sees    = {cfg.embeddings.model}')
print('sanity: OK')

VSTASH_CONFIG = /tmp/merken_colab_sm423ymi/vstash.toml
vstash sees    = sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
sanity: OK


In [3]:
# 3. Sanity run — n=3 questions, should finish in under 2 min.
import os, time, datetime
os.chdir('/content/merken')

commit = !git rev-parse HEAD
branch = !git rev-parse --abbrev-ref HEAD
print(f'commit: {commit[0]}')
print(f'branch: {branch[0]}')

!VSTASH_CONFIG=$VSTASH_CONFIG python -m experiments.retrieval.longmemeval.runner \
    --subset longmemeval_s \
    --questions 3 \
    --seed 42 \
    --top-k 5 \
    --baseline merken-heuristic

commit: 482025e6c5c4e4a6e3fc4aedba495126bf8119fb
branch: develop
# LongMemEval — 3 questions, top_k=5, subset=longmemeval_s, seed=42
/content/vstash/vstash/embed.py:140: UserWarning: The model ━━━━━━━━━━━━━━━━━ 0/1  0:00:00
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 now uses mean 
pooling instead of CLS embedding. In order to preserve the previous behaviour, 
consider either pinning fastembed version to 0.5.1 or using `add_custom_model` 
functionality.
  _onnx_cache[model_name] = TextEmbedding(model_name=model_name)
Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]
⠼ Embedding ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/1  0:00:01Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Fetching 5 files:  20% 1/5 [00:00<00:01,  2.10it/s]
Fetching 5 files:  40% 2/5 [00:02<00:03,  1.31s/it]
Fetching 5 files: 100% 5/5 [00:02<00:00,  2.11it/s]
m⠦ Embedding ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/1 

In [ ]:
# 4. FULL RUN — n=100, merken-heuristic only. ~45 min on Colab CPU.
start_time = time.time()
start_ts = datetime.datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
print(f'started: {start_ts}')

!VSTASH_CONFIG=$VSTASH_CONFIG python -m experiments.retrieval.longmemeval.runner \
    --subset longmemeval_s \
    --questions 100 \
    --seed 42 \
    --top-k 5 \
    --baseline merken-heuristic

elapsed = time.time() - start_time
print(f'\n=== DONE ===')
print(f'elapsed: {elapsed:.0f}s ({elapsed/60:.1f} min)')
print(f'\nFor RESULTS.md:')
print(f'  date:     {datetime.datetime.utcnow().strftime("%Y-%m-%d")}')
print(f'  commit:   {commit[0][:7]}')
print(f'  baseline: merken-heuristic')
print(f'  embedder: paraphrase-multilingual-MiniLM-L12-v2')
print(f'  n=100, seed=42, subset=longmemeval_s')
print(f'  wall-clock: {elapsed/60:.1f} min')

/tmp/ipykernel_1706/966911981.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_ts = datetime.datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')


started: 2026-04-14T15:49:57Z
